# Parameter Servers
:label:`sec_parameterserver`

As we move from a single GPU to multiple GPUs and then to multiple servers containing multiple GPUs, possibly all spread out across multiple racks and network switches,
our algorithms for distributed and parallel training need to become much more sophisticated. Details matter since different interconnects have very different bandwidth (e.g., NVLink can offer up to 100 GB/s across 6 links in an appropriate setting, PCIe 4.0 (16-lane) offers 32 GB/s, while even high speed 100GbE Ethernet only amounts to 10 GB/s). At the same time it is unreasonable to expect that a statistical modeler be an expert in networking and systems.

The core idea of the parameter server was introduced in :citet:`Smola.Narayanamurthy.2010` in the context of distributed latent variable models. A description of the push and pull semantics then followed in :citet:`Ahmed.Aly.Gonzalez.ea.2012` and a description of the system and an open source library followed in :citet:`Li.Andersen.Park.ea.2014`. In the following we will motivate the components needed for efficiency.


## Data-Parallel Training

Let's review the data parallel training approach to distributed training. We will use this to the exclusion of all others in this section since it is significantly simpler to implement in practice. There are virtually no use cases (besides deep learning on graphs) where any other strategy for parallelism is preferred since GPUs have plenty of memory nowadays. :numref:`fig_parameterserver` describes the variant of data parallelism that we implemented in :numref:`sec_multi_gpu`. The key aspect in it is that the aggregation of gradients occurs on one single GPU (GPU 0) before the updated parameters are rebroadcast to all GPUs.

![Left: single GPU training. Right: a variant of multi-GPU training: (1) we compute loss and gradient, (2) all gradients are aggregated on one GPU, (3) parameter update happens and the parameters are re-distributed to all GPUs.](../img/ps.svg)
:label:`fig_parameterserver`

In retrospect, the decision to aggregate on GPU 0 seems rather ad-hoc. After all, we might just as well aggregate on the CPU. In fact, we could even decide to aggregate some of the parameters on one GPU and some others on another. Provided that the optimization algorithm supports this, there is no real reason for why we could not. For instance, if we have four parameter vectors with associated gradients $\mathbf{g}_1, \ldots, \mathbf{g}_4$ we could aggregate the gradients on one GPU for each $\mathbf{g}_i$ ($i = 1, \ldots, 4$).


This reasoning seems arbitrary and frivolous. After all, the mathematics is the same throughout. However, we are dealing with real physical hardware where different buses have different bandwidth as discussed in :numref:`sec_hardware`.
Consider a real 4-way GPU server as described in :numref:`fig_bw_hierarchy`. If it is particularly well connected, it might have a 100 GbE network card. More typical numbers are in the 1--10 GbE range with an effective bandwidth of 100 MB/s to 1 GB/s.
Since the CPUs have too few PCIe lanes to connect to all GPUs directly (e.g., consumer-grade Intel CPUs have 24 lanes) we need a [multiplexer](https://www.broadcom.com/products/pcie-switches-bridges/pcie-switches). The bandwidth from the CPU on a 16x Gen3 link is 16 GB/s. This is also the speed at which *each* of the GPUs is connected to the switch. This means that it is more effective to communicate between the devices.

![A 4-way GPU server.](../img/bw-hierarchy.svg)
:label:`fig_bw_hierarchy`

For the sake of the argument let's assume that the gradients are of 160 MB. In this case it takes 30 ms to send the gradients from all 3 remaining GPUs to the fourth one (each transfer takes 10 ms = 160 MB / 16 GB/s). Adding another 30 ms to transmit the weight vectors back we arrive at a total of 60 ms.
If we send all data to the CPU we incur a penalty of 40 ms since *each* of the four GPUs needs to send the data to the CPU, yielding a total of 80 ms. Lastly assume that we are able to split the gradients into 4 parts of 40 MB each. Now we can aggregate each of the parts on a different GPU *simultaneously* since the PCIe switch offers a full-bandwidth operation between all links. Instead of 30 ms this takes 7.5 ms, yielding a total of 15 ms for a synchronization operation. In short, depending on how we synchronize parameters the same operation can take anywhere from 15 ms to 80 ms. :numref:`fig_ps_distributed` depicts the different strategies for exchanging parameters.

![Parameter synchronization strategies.](../img/ps-distributed.svg)
:label:`fig_ps_distributed`

Note that we have yet another tool at our disposal when it comes to improving performance: in a deep network it takes some time to compute all gradients from the top to the bottom. We can begin synchronizing gradients for some parameter groups even while we are still busy computing them for others. See e.g., :citet:`Sergeev.Del-Balso.2018` for details on how to do this in [Horovod](https://github.com/horovod/horovod).

## Ring Synchronization

When it comes to synchronization on modern deep learning hardware we often encounter significantly bespoke network connectivity. For instance, the AWS p3.16xlarge and NVIDIA DGX-2 instances share the connectivity structure of :numref:`fig_nvlink`. Each GPU connects to a host CPU via a PCIe link which operates at best at 16 GB/s. Additionally each GPU also has 6 NVLink connections, each of which is capable of transferring 300 Gbit/s bidirectionally. This amounts to around 18 GB/s per link per direction. In short, the aggregate NVLink bandwidth is significantly higher than the PCIe bandwidth. The question is how to use it most efficiently.

![NVLink connectivity on 8  V100 GPU servers (image courtesy of NVIDIA).](../img/nvlink.svg)
:label:`fig_nvlink`

It turns out that the optimal synchronization strategy is to decompose the network into two rings and to use them to synchronize data directly :cite:`Wang.Li.Liberty.ea.2018`. :numref:`fig_nvlink_twoloop` illustrates that the network can be decomposed into one ring (1-2-3-4-5-6-7-8-1) with double NVLink bandwidth and into one (1-4-6-3-5-8-2-7-1) with regular bandwidth. Designing an efficient synchronization protocol in this case is nontrivial.

![Decomposition of the NVLink network into two rings.](../img/nvlink-twoloop.svg)
:label:`fig_nvlink_twoloop`


Consider the following thought experiment: given a ring of $n$ computing nodes (or GPUs) we can send gradients from the first to the second node. There it is added to the local gradient and sent on to the third node, and so on. After $n-1$ steps the aggregate gradient can be found in the last-visited node. That is, the time to aggregate gradients grows linearly with the number of nodes. But if we do this the algorithm is quite inefficient. After all, at any time there is only one of the nodes communicating. What if we broke the gradients into $n$ chunks and started synchronizing chunk $i$ starting at node $i$?
Since each chunk is of size $1/n$ the total time is now $(n-1)/n \approx 1$. In other words, the time spent to aggregate gradients *does not grow* as we increase the size of the ring. This is quite an astonishing result. :numref:`fig_ringsync` illustrates the sequence of steps on $n=4$ nodes.

![Ring synchronization across 4 nodes. Each node starts transmitting parts of gradients to its left neighbor until the assembled gradient can be found in its right neighbor.](../img/ringsync.svg)
:label:`fig_ringsync`

If we use the same example of synchronizing 160 MB across 8 V100 GPUs we arrive at approximately $2 \cdot 160 \textrm{MB} / (3 \cdot 18 \textrm{GB/s}) \approx 6 \textrm{ms}$. This is better than using the PCIe bus, even though we are now using 8 GPUs. Note that in practice these numbers are a bit worse, since deep learning frameworks often fail to assemble communication into large burst transfers.

Note that there is a common misconception that ring synchronization is fundamentally different from other synchronization algorithms. The only difference is that the synchronization path is somewhat more elaborate when compared with a simple tree.

## Multi-Machine Training

Distributed training on multiple machines adds a further challenge: we need to communicate with servers that are only connected across a comparatively lower bandwidth fabric that can be over an order of magnitude slower in some cases.
Synchronization across devices is tricky. After all, different machines running training code will have subtly different speed. Hence we need to *synchronize* them if we want to use synchronous distributed optimization. :numref:`fig_ps_multimachine` illustrates how distributed parallel training occurs.

1. A (different) batch of data is read on each machine, split across multiple GPUs and transferred to GPU memory. There predictions and gradients are computed on each GPU batch separately.
2. The gradients from all local GPUs are aggregated on one GPU (or parts of it are aggregated over different GPUs).
3. The gradients are sent to the CPUs.
4. The CPUs send the gradients to a central parameter server which aggregates all the gradients.
5. The aggregate gradients are then used to update the parameters and the updated parameters are broadcast back to the individual CPUs.
6. The information is sent to one (or multiple) GPUs.
7. The updated parameters are spread across all GPUs.

![Multi-machine multi-GPU distributed parallel training.](../img/ps-multimachine.svg)
:label:`fig_ps_multimachine`

Each of these operations seems rather straightforward. And, indeed, they can be carried out efficiently *within* a single machine. Once we look at multiple machines, though, we can see that the central parameter server becomes the bottleneck. After all, the bandwidth per server is limited, hence for $m$ workers the time it takes to send all gradients to the server is $\mathcal{O}(m)$. We can break through this barrier by increasing the number of servers to $n$. At this point each server only needs to store $\mathcal{O}(1/n)$ of the parameters, hence the total time for updates and optimization becomes $\mathcal{O}(m/n)$.
Matching both numbers yields constant scaling regardless of how many workers we are dealing with. In practice we use the *same* machines both as workers and as servers. :numref:`fig_ps_multips` illustrates the design (see also :cite:`Li.Andersen.Park.ea.2014` for details).
In particular, ensuring that multiple machines work without unreasonable delays is nontrivial. 

![Top: a single parameter server is a bottleneck since its bandwidth is finite. Bottom: multiple parameter servers store parts of the parameters with aggregate bandwidth.](../img/ps-multips.svg)
:label:`fig_ps_multips`

## Key--Value Stores

Implementing the steps required for distributed multi-GPU training in practice is nontrivial.
This is why it pays to use a common abstraction, namely that of a *key--value store* with redefined update semantics.


Across many workers and many GPUs the computation for gradient $i$ can be defined as

$$\mathbf{g}_{i} = \sum_{k \in \textrm{workers}} \sum_{j \in \textrm{GPUs}} \mathbf{g}_{ijk},$$

where $\mathbf{g}_{ijk}$ is part of gradient $i$ split on GPU $j$ of worker $k$.
The key aspect in this operation is that it is a *commutative reduction*, that is, it turns many vectors into one and the order in which the operation is applied does not matter. This is great for our purposes since we do not (need to) have fine grained control over when which gradient is received. Besides, note that this operation is independent among different $i$.

This allows us to define the following two operations: *push*, which accumulates gradients, and *pull*, which retrieves aggregate gradients. Since we have many different sets of gradients (after all, we have many layers), we need to index the gradients with a key $i$. This similarity to key--value stores, such as the one introduced in Dynamo
:cite:`DeCandia.Hastorun.Jampani.ea.2007` is not by coincidence. They, too, satisfy many similar characteristics, in particular when it comes to distributing the parameters across multiple servers.


The push and pull operations for key-value stores are described as follows:

* **push(key, value)** sends a particular gradient (the value) from a worker to a common storage. There the value is aggregated, e.g., by summing it up.
* **pull(key, value)** retrieves an aggregate value from common storage, e.g., after combining the gradients from all workers.

By hiding all the complexity about synchronization behind a simple push and pull operation we can decouple the concerns of statistical modelers who want to be able to express optimization in simple terms and the system engineers who need to deal with the complexity inherent in distributed synchronization.

## Summary

* Synchronization needs to be highly adaptive to specific network infrastructure and connectivity within a server. This can make a significant difference to the time it takes to synchronize.
* Ring-synchronization can be optimal for p3 and DGX-2 servers. For others possibly not so much.
* A hierarchical synchronization strategy works well when adding multiple parameter servers for increased bandwidth.


## Exercises

1. Can you increase the ring synchronization even further? Hint: you can send messages in both directions.
1. Is it possible to allow asynchronous communication (while computation is still ongoing)? How does it affect performance?
1. What if we lost a server during a long-running computation? How can we design a *fault tolerance* mechanism to avoid restarting the computation fully?


[Discussions](https://discuss.d2l.ai/t/366)


# Advanced Distributed Training Strategies

## Problem 1: Bi-directional Ring Synchronization

**Problem:** Can you increase the ring synchronization even further? Hint: you can send messages in both directions.

**Solution:**

Standard ring allreduce algorithms operate by passing gradients in a single direction around a logical ring of nodes. We can improve this by implementing a bi-directional ring synchronization, often called a bi-directional ring allreduce.

### Standard Ring Allreduce Recap

In the standard ring allreduce algorithm with $n$ nodes:
- Each node initially has its own local gradient $G_i$
- The algorithm takes $2(n-1)$ steps to complete
- In the first $n-1$ steps (scatter-reduce phase), partial sums are built up
- In the next $n-1$ steps (allgather phase), completed chunks are distributed

### Bi-directional Ring Allreduce

In a bi-directional approach:
1. We split the gradient tensor into $2k$ chunks (where $k$ is typically chosen to be $n/2$ or similar)
2. We run two ring synchronizations in parallel:
   - A "clockwise" ring sending chunks 0 to $k-1$
   - A "counterclockwise" ring sending chunks $k$ to $2k-1$

#### Algorithm Description

1. **Initialization**:
   - Each node $i$ splits its gradient $G_i$ into $2k$ chunks: $[G_i^0, G_i^1, ..., G_i^{2k-1}]$
   - Define clockwise neighbor of node $i$ as node $(i+1) \mod n$
   - Define counterclockwise neighbor of node $i$ as node $(i-1+n) \mod n$

2. **Bi-directional Scatter-Reduce Phase** (takes $n-1$ steps):
   - In each step $s$ from 0 to $n-2$:
     - Node $i$ sends chunk $G_i^{(i-s) \mod k}$ to its clockwise neighbor
     - Node $i$ sends chunk $G_i^{k+(i+s) \mod k}$ to its counterclockwise neighbor
     - Node $i$ receives chunks from both neighbors and adds them to its local chunks

3. **Bi-directional Allgather Phase** (takes $n-1$ steps):
   - In each step $s$ from 0 to $n-2$:
     - Node $i$ sends chunk $G_i^{(i-s-1+n) \mod k}$ to its clockwise neighbor
     - Node $i$ sends chunk $G_i^{k+(i+s+1) \mod k}$ to its counterclockwise neighbor
     - Node $i$ receives chunks from both neighbors and updates its local chunks

### Performance Analysis

1. **Theoretical Improvement**:
   - The regular ring allreduce takes $2(n-1)$ steps
   - The bi-directional ring allreduce still takes $2(n-1)$ steps, but each step processes twice as much data
   - This leads to approximately 2× theoretical speedup

2. **Bandwidth Utilization**:
   - Bi-directional communication utilizes both incoming and outgoing bandwidth simultaneously
   - Most modern network interfaces can support full-duplex communication, so this leverages otherwise unused bandwidth

3. **Practical Considerations**:
   - The speedup is most noticeable when network bandwidth is the bottleneck
   - In bandwidth-constrained environments, we approach the full 2× speedup
   - In latency-dominated scenarios (small gradients or high-latency networks), the improvement is less significant

### Implementation Considerations

```python
def bidirectional_ring_allreduce(local_gradient, world_size, rank):
    # Split gradient into 2*world_size chunks for better balance
    num_chunks = 2 * world_size
    chunks = split_tensor(local_gradient, num_chunks)
    
    # Define neighbors
    clockwise_neighbor = (rank + 1) % world_size
    counterclockwise_neighbor = (rank - 1) % world_size
    
    # Scatter-reduce phase
    for step in range(world_size - 1):
        # Determine which chunks to send in each direction
        clockwise_chunk_idx = (rank - step) % (num_chunks // 2)
        counterclockwise_chunk_idx = (num_chunks // 2) + (rank + step) % (num_chunks // 2)
        
        # Send and receive in both directions
        send_clockwise_async(chunks[clockwise_chunk_idx], clockwise_neighbor)
        send_counterclockwise_async(chunks[counterclockwise_chunk_idx], counterclockwise_neighbor)
        
        recv_from_clockwise = receive_from_counterclockwise(counterclockwise_neighbor)
        recv_from_counterclockwise = receive_from_clockwise(clockwise_neighbor)
        
        # Update local chunks with received data
        clockwise_recv_idx = (clockwise_chunk_idx - 1) % (num_chunks // 2)
        counterclockwise_recv_idx = (num_chunks // 2) + (counterclockwise_chunk_idx + 1) % (num_chunks // 2)
        
        chunks[clockwise_recv_idx] += recv_from_counterclockwise
        chunks[counterclockwise_recv_idx] += recv_from_clockwise
    
    # Allgather phase (similar structure but just copying, not adding)
    # ...similar implementation with appropriate index adjustments...
    
    return reassemble_tensor(chunks)
```

### Optimization Extensions

1. **Chunk Optimization**:
   - Instead of creating $2n$ chunks, we can create $2k$ chunks where $k$ is optimized based on tensor size and network characteristics
   - Larger chunks reduce overhead but may unbalance the workload

2. **Hierarchical Approach**:
   - For multi-rack deployments, we can use bi-directional rings within racks
   - Then use a separate algorithm for inter-rack communication

**Intuition:** Standard ring allreduce is like passing a baton in one direction around a circle. Bi-directional ring allreduce is like passing two batons in opposite directions simultaneously, allowing twice as much work to be done in the same time.

## Problem 2: Asynchronous Communication

**Problem:** Is it possible to allow asynchronous communication (while computation is still ongoing)? How does it affect performance?

**Solution:**

Yes, it's possible to implement asynchronous communication that overlaps with computation, significantly improving overall performance in distributed training. This technique is often called computation-communication overlap.

### Synchronous vs. Asynchronous Communication

In standard synchronous communication:
1. Compute forward and backward passes
2. **Wait** until computation is complete
3. Communicate gradients between nodes
4. **Wait** until communication is complete
5. Update model parameters
6. Begin next iteration

With asynchronous communication:
1. Compute forward and backward passes **for layer L**
2. **Immediately** start communicating gradients for layer L
3. **In parallel**, continue backward computation for layers L-1, L-2, etc.
4. Apply updates to layer L once both its communication and the backward passes for all layers are complete

### Implementation Approaches

#### 1. Layer-wise Asynchronous Communication

This approach, often called "wait-free backpropagation" or "gradient splitting," works as follows:

```python
def train_with_async_communication(model, optimizer, data_loader):
    for inputs, targets in data_loader:
        # Forward pass
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        
        # Initialize communication handles
        comm_handles = []
        
        # Backward pass with overlapped communication
        loss.backward(retain_graph=True)  # Ensure graph is kept for layer-wise gradient access
        
        # Process each layer's gradients as soon as they're available
        for layer in reversed(model.layers):
            # Compute gradients for this layer (if not already computed by autograd)
            if hasattr(layer, 'weight') and layer.weight.grad is None:
                # Perform local backward pass for this layer
                torch.autograd.backward(loss, retain_graph=True, 
                                       grad_tensors=[...],  # Appropriate gradient tensors
                                       inputs=[layer.weight, layer.bias])
            
            # Start asynchronous communication for this layer's gradients
            if hasattr(layer, 'weight') and layer.weight.grad is not None:
                handle = allreduce_async(layer.weight.grad)
                comm_handles.append((handle, layer.weight.grad))
                
                if hasattr(layer, 'bias') and layer.bias.grad is not None:
                    handle = allreduce_async(layer.bias.grad)
                    comm_handles.append((handle, layer.bias.grad))
        
        # Wait for all communications to complete
        for handle, grad_tensor in comm_handles:
            handle.wait()
            # Division by world_size happens in-place
            grad_tensor.div_(world_size)
        
        # Apply optimizer update
        optimizer.step()
        optimizer.zero_grad()
```

#### 2. Tensor Splitting for Finer-Grained Overlap

For very large layers (like embeddings), further splitting individual tensor gradients:

```python
def allreduce_split_async(tensor, num_splits=4):
    """Split a tensor into chunks and allreduce them asynchronously."""
    splits = torch.split(tensor, tensor.numel() // num_splits)
    handles = [allreduce_async(split) for split in splits]
    return handles, splits

# In training loop:
for large_tensor in large_tensors:
    handles, splits = allreduce_split_async(large_tensor.grad)
    # Continue computation for other layers
    # ...
    # Wait for completion
    for handle, split in zip(handles, splits):
        handle.wait()
    # Reassemble if needed
```

#### 3. CUDA Stream-Based Approach

Using separate CUDA streams for computation and communication:

```python
# Create separate streams
compute_stream = torch.cuda.Stream()
comm_stream = torch.cuda.Stream()

# In training loop
with torch.cuda.stream(compute_stream):
    # Perform backward pass
    loss.backward()

# Start communication in comm_stream as soon as gradients are ready
for param in model.parameters():
    if param.requires_grad and param.grad is not None:
        # Record event in compute stream
        event = torch.cuda.Event()
        event.record(compute_stream)
        
        # Wait for event in comm stream before starting communication
        with torch.cuda.stream(comm_stream):
            event.wait()
            allreduce_async(param.grad)

# Ensure all communications complete before optimizer step
comm_stream.synchronize()
optimizer.step()
```

### Performance Impact Analysis

The performance impact of asynchronous communication depends on several factors:

1. **Computation-to-Communication Ratio**:
   - High ratio (compute-bound): Less significant speedup (10-30%)
   - Low ratio (communication-bound): Major speedup (up to 80-90% reduction in overall iteration time)

2. **Model Architecture Factors**:
   - Large models with many parameters benefit more
   - Models with computationally intensive layers toward the end benefit more from layer-wise approaches

3. **Hardware Considerations**:
   - GPUs with unified memory controllers may see less benefit due to resource contention
   - InfiniBand/high-speed networks with dedicated DMA engines benefit more
   - RDMA-capable networks (like InfiniBand or RoCE) provide the best speedups

4. **Scaling Behavior**:
   - Benefits increase with more nodes as communication becomes a larger bottleneck
   - For N nodes, synchronous algorithms have communication scaling with O(log N) or O(N)
   - Asynchronous approaches can maintain near-constant computation time regardless of node count

### Potential Drawbacks

1. **Implementation Complexity**:
   - Significantly more complex than synchronous approaches
   - Debugging and performance tuning become more challenging

2. **Memory Usage**:
   - May require additional memory for buffering gradients
   - Can require keeping activation tensors in memory longer

3. **Hardware Requirements**:
   - Requires network hardware that supports asynchronous operations
   - Benefits most from RDMA-capable networks

4. **Numerical Effects**:
   - With true asynchronous SGD (not just communication), there can be gradient staleness
   - This may affect convergence properties or final accuracy

**Intuition:** Asynchronous communication is like starting to explain your thoughts before you've fully formed them in your mind. As you continue thinking, you're simultaneously communicating the parts you've already figured out. It's more efficient than waiting to formulate your entire thought before beginning to speak.

## Problem 3: Fault Tolerance Mechanisms

**Problem:** What if we lost a server during a long-running computation? How can we design a *fault tolerance* mechanism to avoid restarting the computation fully?

**Solution:**

In distributed deep learning, fault tolerance is critical for long-running jobs that might take days or weeks. Several mechanisms can be implemented to handle server failures without requiring a complete restart.

### 1. Checkpoint-Based Recovery

The most common approach is periodic checkpointing, where the training state is saved to persistent storage at regular intervals.

#### Basic Checkpoint Implementation

```python
def train_with_checkpointing(model, optimizer, dataset, num_epochs, 
                           checkpoint_interval=100, checkpoint_dir="./checkpoints"):
    global_step = 0
    start_epoch = 0
    
    # Try to load latest checkpoint if exists
    latest_checkpoint = find_latest_checkpoint(checkpoint_dir)
    if latest_checkpoint:
        state = torch.load(latest_checkpoint)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        global_step = state['global_step']
        start_epoch = state['epoch'] + 1
        print(f"Restored from checkpoint at step {global_step}, epoch {start_epoch}")
    
    for epoch in range(start_epoch, num_epochs):
        for batch_idx, (data, target) in enumerate(dataset):
            # Training step
            optimizer.zero_grad()
            output = model(data)
            loss = loss_function(output, target)
            loss.backward()
            optimizer.step()
            
            global_step += 1
            
            # Save checkpoint periodically
            if global_step % checkpoint_interval == 0:
                checkpoint_path = f"{checkpoint_dir}/checkpoint-{global_step}.pt"
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step,
                    'model': model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'loss': loss.item()
                }, checkpoint_path)
                print(f"Saved checkpoint at step {global_step}, epoch {epoch}")
```

#### Advanced Checkpointing Strategies

1. **Hierarchical Checkpointing**:
   - Frequent lightweight checkpoints in-memory or on local SSDs
   - Less frequent but more comprehensive checkpoints to remote storage
   - Example pattern: Local checkpoints every 100 steps, remote every 1000 steps

2. **Asynchronous Checkpointing**:
   - Continue training while checkpoint is written in background
   - Requires careful state management to ensure consistency

```python
def save_checkpoint_async(state, path):
    # Create a copy of tensors to avoid in-place modifications during save
    state_copy = {
        'epoch': state['epoch'],
        'global_step': state['global_step'],
        'model': {k: v.detach().cpu().clone() for k, v in state['model'].items()},
        'optimizer': state['optimizer'],  # May need similar detaching/cloning
        'loss': state['loss']
    }
    
    # Save in a separate thread
    thread = threading.Thread(
        target=lambda: torch.save(state_copy, path)
    )
    thread.start()
    return thread  # Return thread handle for potential waiting
```

### 2. Distributed Checkpoint Coordination

In a distributed setting, checkpoint coordination is crucial to ensure consistency across nodes.

1. **Synchronous Multi-Node Checkpointing**:
   - All nodes pause at the same step to create consistent checkpoints
   - One node (typically rank 0) is responsible for initiating the checkpoint process

```python
def distributed_checkpoint(model, optimizer, epoch, step, dir="./checkpoints"):
    # Synchronize all nodes
    torch.distributed.barrier()
    
    # Only rank 0 saves the optimizer state and training metadata
    if torch.distributed.get_rank() == 0:
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'optimizer': optimizer.state_dict()
        }
        torch.save(checkpoint, f"{dir}/metadata-{step}.pt")
    
    # Each node saves its model shard or replica
    model_state = model.state_dict()
    torch.save(model_state, 
              f"{dir}/model-rank{torch.distributed.get_rank()}-{step}.pt")
    
    # Synchronize after saving
    torch.distributed.barrier()
```

2. **Elastic Synchronization Point**:
   - Rather than checkpointing at the exact same iteration, ensure all ranks reach a specific synchronization point

### 3. Model Replication and Hot Standby

For mission-critical training, maintaining redundant copies of the model and optimizer state:

1. **N+K Replication**:
   - For N training nodes, maintain K additional standby nodes
   - Each parameter server replicates its state to one or more standby servers

2. **Implementation Strategy**:
   - Primary nodes perform training
   - Replica nodes receive regular state updates but don't participate in training
   - Upon failure detection, replica takes over for the failed node

```python
class ReplicatedParameterServer:
    def __init__(self, rank, world_size, model, replicas=[]):
        self.rank = rank
        self.world_size = world_size
        self.model = model
        self.replica_ranks = replicas
        self.is_replica = False
        self.primary_rank = None
        
    def sync_to_replicas(self):
        if not self.is_replica:
            # Primary sends its state to replicas
            for replica_rank in self.replica_ranks:
                for name, param in self.model.named_parameters():
                    torch.distributed.send(param.data, dst=replica_rank)
    
    def become_primary(self, failed_rank):
        if self.is_replica and self.primary_rank == failed_rank:
            self.is_replica = False
            print(f"Rank {self.rank} taking over for failed rank {failed_rank}")
            # Update distributed group membership
            # ...
```

### 4. Heartbeat and Failure Detection

A critical component of any fault-tolerant system is reliable failure detection:

```python
class HeartbeatMonitor:
    def __init__(self, rank, world_size, timeout_sec=30):
        self.rank = rank
        self.world_size = world_size
        self.timeout_sec = timeout_sec
        self.last_heartbeats = {r: time.time() for r in range(world_size) if r != rank}
        self.alive_nodes = set(range(world_size))
        self.lock = threading.Lock()
        self.heartbeat_thread = threading.Thread(target=self._heartbeat_loop)
        self.monitor_thread = threading.Thread(target=self._monitor_loop)
        
    def start(self):
        self.heartbeat_thread.start()
        self.monitor_thread.start()
        
    def _send_heartbeat(self):
        message = torch.tensor([self.rank, time.time()], dtype=torch.float64)
        for r in self.alive_nodes:
            if r != self.rank:
                try:
                    torch.distributed.send(message, dst=r)
                except Exception:
                    pass  # Handle send failure gracefully
    
    def _receive_heartbeats(self):
        for r in list(self.alive_nodes):
            if r != self.rank:
                try:
                    message = torch.zeros(2, dtype=torch.float64)
                    status = torch.distributed.recv(message, src=r, timeout=0)
                    if status:
                        with self.lock:
                            self.last_heartbeats[r] = message[1].item()
                except Exception:
                    pass  # Non-blocking receive
    
    def _check_timeouts(self):
        now = time.time()
        failed_nodes = []
        
        with self.lock:
            for r in list(self.alive_nodes):
                if r != self.rank and now - self.last_heartbeats.get(r, 0) > self.timeout_sec:
                    failed_nodes.append(r)
                    self.alive_nodes.remove(r)
        
        for failed_rank in failed_nodes:
            self._handle_node_failure(failed_rank)
    
    def _handle_node_failure(self, failed_rank):
        print(f"Node {failed_rank} has failed. Initiating recovery procedure.")
        # Recovery actions - reassign work, activate standby, restore from checkpoint, etc.
        # ...
```

### 5. Gradient Checkpointing for Memory-Efficient Recovery

For very large models, storing full model state may be prohibitive. Gradient checkpointing can be used to trade computation for memory:

```python
def train_with_gradient_checkpointing(model, optimizer, dataset):
    # Enable gradient checkpointing
    model.gradient_checkpointing_enable()
    
    for data, target in dataset:
        optimizer.zero_grad()
        
        # Forward pass with checkpoints
        output = model(data)
        loss = loss_function(output, target)
        
        # Backward pass will recompute activations as needed
        loss.backward()
        optimizer.step()
```

### 6. Zero Redundancy Optimizer (ZeRO) with Recovery

ZeRO-based approaches can be extended for fault tolerance:

```python
class ZeROWithRecovery:
    def __init__(self, model, world_size, rank):
        self.model = model
        self.world_size = world_size
        self.rank = rank
        self.parameter_shards = self._partition_parameters()
        
    def _partition_parameters(self):
        # Assign each parameter to a rank based on parameter index
        shards = {r: [] for r in range(self.world_size)}
        for i, p in enumerate(self.model.parameters()):
            shards[i % self.world_size].append(p)
        return shards
    
    def backup_my_shard(self, path):
        """Save this rank's parameter shard to disk."""
        my_shard = self.parameter_shards[self.rank]
        shard_state = {i: p.data.clone() for i, p in enumerate(my_shard)}
        torch.save(shard_state, f"{path}/shard-{self.rank}.pt")
    
    def restore_from_backup(self, path, failed_rank=None):
        """Restore parameters from backup."""
        if failed_rank is not None:
            # Only restore the failed rank's shard
            shard_path = f"{path}/shard-{failed_rank}.pt"
            if os.path.exists(shard_path):
                shard_state = torch.load(shard_path)
                # Logic to redistribute the failed rank's parameters
                # ...
        else:
            # Restore all shards (full restart)
            for r in range(self.world_size):
                shard_path = f"{path}/shard-{r}.pt"
                if os.path.exists(shard_path):
                    # ...restore logic...
```

### 7. Practical Implementation Considerations

1. **Recovery Coordination**:
   - Designate a coordination server or use consensus algorithms
   - Ensure all nodes agree on the recovery procedure

2. **State Consistency**:
   - Address potential inconsistencies in optimizer states
   - Consider stateless optimizers for easier recovery

3. **Automatic Restart with Elastic Training**:
   - Configure framework to automatically restart failed nodes
   - Use elastic training frameworks that support dynamic world sizes

4. **Progressive Validation**:
   - Validate checkpoint integrity before full restoration
   - Use checksums or digests to detect corruption

**Intuition:** Fault tolerance in distributed training is like having a team of skydivers where each person carries a piece of valuable equipment. If one person's parachute fails, you don't want to lose their equipment and restart the entire mission. Instead, you have backup parachutes (replicas), frequent photos of the equipment configuration (checkpoints), and a plan for redistributing the equipment if someone can't complete the jump (recovery procedures).